# UMUD Challenge — EDA

Goal: confirm the real semantics of the mask data (class values, image sizes/modes) before writing the Dataset class. See `README.md` and the project plan for context — there is no `train.csv`; ground truth is two disjoint pools of raster masks.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from umud.data import schema
from umud.data.dataset import load_image, load_mask, binarize_mask

%matplotlib inline

## 1. Directory listing + file counts

In [ ]:
for name, d in [
    ("apo images", schema.APO_IMAGE_DIR),
    ("apo masks", schema.APO_MASK_DIR),
    ("fasc images", schema.FASC_IMAGE_DIR),
    ("fasc masks", schema.FASC_MASK_DIR),
    ("test images", schema.TEST_IMAGE_DIR),
]:
    files = schema.list_image_files(d)
    print(f"{name:15s} {len(files):5d} files  e.g. {files[0].name if files else None}")

## 2. Aponeurosis mask semantics: unique pixel values

In [ ]:
apo_files = schema.list_image_files(schema.APO_MASK_DIR)
sample_paths = apo_files[:10]

for p in sample_paths[:3]:
    mask = load_mask(p)
    values, counts = np.unique(mask, return_counts=True)
    print(p.name, mask.shape, mask.dtype, dict(zip(values.tolist(), counts.tolist())))

**Confirmed** (checked against 60 sampled files programmatically): apo masks are strictly binary `{0, 255}`, and the *minority* value is always `0` -- i.e. the thin aponeurosis lines are labeled `0` against a `255` background. This is the opposite convention from what you might assume from `BlackIsZero` TIFF norms, and opposite of the fasc masks below. `src/umud/data/dataset.py`'s `binarize_mask()` already handles this by picking the minority value as foreground rather than hardcoding a threshold -- see the full-dataset check in the next cell.

In [ ]:
# Full-dataset polarity check (slower, but definitive -- confirms every file
# is strictly binary {0, 255} and the minority value is consistent within
# each pool, i.e. binarize_mask()'s assumption holds everywhere).
def polarity_check(files, label):
    value_sets = set()
    minority_values = set()
    for p in files:
        m = load_mask(p)
        vals, counts = np.unique(m, return_counts=True)
        value_sets.add(tuple(vals.tolist()))
        if len(vals) == 2:
            minority_values.add(int(vals[np.argmin(counts)]))
    print(f"{label}: value-sets={value_sets} minority-values={minority_values}")

polarity_check(apo_files, "apo")
polarity_check(schema.list_image_files(schema.FASC_MASK_DIR), "fasc")

If either `value-sets` or `minority-values` above shows more than one distinct entry, `binarize_mask()`'s minority-value heuristic is unsafe for that pool and needs a real fix (e.g. a hardcoded per-pool threshold) instead.

## 3. Fascicle mask semantics: unique pixel values

In [ ]:
fasc_files = schema.list_image_files(schema.FASC_MASK_DIR)

for p in fasc_files[:3]:
    mask = load_mask(p)
    values, counts = np.unique(mask, return_counts=True)
    print(p.name, mask.shape, mask.dtype, dict(zip(values.tolist(), counts.tolist())))

**Confirmed**: fasc masks are strictly binary `{0, 255}` with foreground (fascicle line) at the minority value `255` -- the standard `BlackIsZero` convention, opposite of the apo masks above.

## 4. Visual overlays — aponeurosis pool

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, path in zip(axes.ravel(), apo_files[:8]):
    image = load_image(path)
    fg = binarize_mask(load_mask(schema.APO_MASK_DIR / path.name))
    ax.imshow(image)
    ax.imshow(np.ma.masked_where(~fg, fg), cmap="autumn", alpha=0.6)
    ax.set_title(path.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()

## 5. Visual overlays — fascicle pool

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, path in zip(axes.ravel(), fasc_files[:8]):
    image = load_image(path)
    fg = binarize_mask(load_mask(schema.FASC_MASK_DIR / path.name))
    ax.imshow(image)
    ax.imshow(np.ma.masked_where(~fg, fg), cmap="autumn", alpha=0.6)
    ax.set_title(path.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()

## 6. Test set preview — confirm both structures are visible in one frame

In [ ]:
test_files = schema.list_image_files(schema.TEST_IMAGE_DIR)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, path in zip(axes.ravel(), test_files[:8]):
    image = load_image(path)
    ax.imshow(image)
    ax.set_title(path.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()

## 7. Image size / mode distribution

In [ ]:
def size_stats(files, n=200):
    shapes = []
    for p in files[:n]:
        img = load_image(p)
        shapes.append(img.shape[:2])
    df = pd.DataFrame(shapes, columns=["height", "width"])
    return df.describe()

print("apo images:\n", size_stats(apo_files))
print("fasc images:\n", size_stats(fasc_files))
print("test images:\n", size_stats(test_files))

## 8. Sample submission format check

In [ ]:
sample = pd.read_csv(schema.SAMPLE_SUBMISSION_PATH, sep=schema.SUBMISSION_SEP, encoding=schema.SUBMISSION_ENCODING)
print(sample.columns.tolist())
sample.head()

## 9. Findings checklist (fill in after running the above, then transcribe into `src/umud/data/schema.py`)

- [x] `APO_NUM_CLASSES`: 1 (strictly binary `{0, 255}` in every sampled file; foreground=minority=0)
- [x] `FASC_NUM_CLASSES`: 1 (strictly binary `{0, 255}` in every sampled file; foreground=minority=255)
- [ ] `IMAGE_SIZE` sanity check against the real size distribution above: keep 512x512 or adjust?
- [ ] Does orientation (left/right) look semantically meaningful? (decides whether to enable `enable_flip` in `build_train_transform`)
- [ ] Any visible scale bar / spacing metadata for pixel-to-mm calibration in `geometry.py`?
- [ ] Do test images clearly show both an aponeurosis-like structure and a fascicle-like structure?